# Human Activity Recognition from Smartphone Sensors
## Stages 1, 2 & 3: Problem Definition · Data Collection · Preprocessing

**Dataset:** UCI HAR Dataset  
**Goal:** Classify physical activities (walking, sitting, standing, running, stair climbing)  
from accelerometer and gyroscope data.

---

# Stage 1 – Problem Definition & Literature Review <a id='stage1'></a>

## 1.1 Problem Statement

**Human Activity Recognition (HAR)** is the task of automatically identifying what physical activity a person is performing based on sensor readings collected from a smartphone worn on the body (typically at the waist).

### Activities to Classify
| Label | Activity |
|-------|----------|
| 1 | Walking |
| 2 | Walking Upstairs |
| 3 | Walking Downstairs |
| 4 | Sitting |
| 5 | Standing |
| 6 | Laying |

### Sensors Used
- **Accelerometer** – measures linear acceleration in 3 axes (X, Y, Z)
- **Gyroscope** – measures angular velocity in 3 axes

### Why This Matters
- **Healthcare:** Remote patient monitoring, rehabilitation tracking
- **Fitness:** Automatic workout logging on smartphones and smartwatches
- **Elderly Care:** Fall detection and daily activity monitoring
- **Sports Science:** Athlete performance analysis

## 1.2 Project Objectives

1. **Feature Engineering** – Extract time-domain and frequency-domain statistical features from sliding windows of raw sensor signals.
2. **Classical ML Models** – Train and tune **SVM** and **Random Forest** classifiers on handcrafted features.
3. **Deep Learning Model** – Train a **LSTM** (Long Short-Term Memory) network directly on raw sequential sensor data.
4. **Comparison** – Compare handcrafted feature pipelines vs. end-to-end deep learning.
5. **Generalization** – Evaluate all models on held-out **unseen subjects** (person-independent test split).

## 1.3 Literature Review

### Foundational Work

**Anguita et al. (2013)** introduced the UCI HAR dataset. Their pipeline used a **561-dimensional feature vector** extracted from 2.56-second sliding windows (50% overlap) over triaxial accelerometer and gyroscope signals. They trained a **Multiclass SVM** and reported **~89% accuracy** on the test set. This established the benchmark for classical ML on HAR.

**Wang et al. (2019)** — *Deep Learning for Sensor-Based Activity Recognition: A Survey* — surveyed deep learning methods including CNN, LSTM, and hybrid CNN-LSTM architectures. Key finding: deep models eliminate the need for manual feature engineering and can learn hierarchical temporal representations, consistently outperforming classical ML by 5–10% on public benchmarks.

**Ordóñez & Roggen (2016)** proposed **DeepConvLSTM**, a deep network combining convolutional and recurrent layers for HAR. Achieved state-of-the-art at the time on Opportunity and PAMAP2 datasets. Demonstrated that raw sensor input + deep architecture matches or beats handcrafted pipelines.

**Ronao & Cho (2016)** applied a **CNN directly** to UCI HAR raw signals. Achieved **94.79% accuracy** — substantially higher than classical SVM on the same dataset, confirming the advantage of learned representations.

### Key Insights from Literature

| Approach | Typical Accuracy (UCI HAR) | Feature Type |
|----------|---------------------------|---------------|
| SVM (561 features) | ~89% | Handcrafted |
| Random Forest | ~91% | Handcrafted |
| LSTM | ~92–93% | Raw sequences |
| CNN | ~94–95% | Raw sequences |
| CNN-LSTM hybrid | ~95–96% | Raw sequences |

### Challenges Identified
- **Subject variability:** Different people perform the same activity differently (gait, pace, phone placement).
- **Transition states:** The boundary between activities (e.g., transitioning from walking to sitting) is ambiguous.
- **Static vs. dynamic activities:** Sitting and standing are easily confused as both produce near-zero acceleration.
- **Generalization:** Models trained on specific subjects may not generalize — person-independent evaluation is essential.

## 1.4 Proposed Pipeline

```
Raw Sensor Data (Accelerometer + Gyroscope)
         │
         ▼
   Sliding Windows (2.56s, 50% overlap)
         │
   ┌─────┴──────┐
   │            │
   ▼            ▼
Handcrafted   Raw Sequences
Features      (for LSTM)
(561-dim)         │
   │              ▼
   │           LSTM Model
   │
   ├──► SVM Classifier
   └──► Random Forest
         │
         ▼
  Evaluation on Unseen Subjects
  (Accuracy, F1, Confusion Matrix)
```

In [3]:
# Stage 1 – Library Imports & Environment Check
import sys
import numpy as np
import pandas as pd
import matplotlib
import sklearn

print("=== Environment Check ===")
print(f"Python  : {sys.version.split()[0]}")
print(f"NumPy   : {np.__version__}")
print(f"Pandas  : {pd.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")

try:
    import tensorflow as tf
    print(f"TensorFlow: {tf.__version__}")
except ImportError:
    print("TensorFlow: not installed (run: pip install tensorflow)")

=== Environment Check ===
Python  : 3.13.6
NumPy   : 2.2.6
Pandas  : 3.0.1
Matplotlib: 3.10.8
Scikit-learn: 1.8.0
TensorFlow: 2.21.0


In [4]:
# ─── Global Configuration ───────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')

# ── Dataset paths (update DATA_ROOT if needed) ──
DATA_ROOT   = 'UCI HAR Dataset'          # top-level folder after unzipping
TRAIN_PATH  = os.path.join(DATA_ROOT, 'train')
TEST_PATH   = os.path.join(DATA_ROOT, 'test')

# ── Signal windowing parameters (matching original paper) ──
WINDOW_SIZE  = 128    # samples per window  (2.56 s @ 50 Hz)
OVERLAP      = 64     # 50% overlap
SAMPLE_RATE  = 50     # Hz

# ── Activity label mapping ──
ACTIVITY_MAP = {
    1: 'Walking',
    2: 'Walking Upstairs',
    3: 'Walking Downstairs',
    4: 'Sitting',
    5: 'Standing',
    6: 'Laying'
}

# ── Sensor channel names ──
SIGNAL_NAMES = [
    'body_acc_x', 'body_acc_y', 'body_acc_z',
    'body_gyro_x', 'body_gyro_y', 'body_gyro_z',
    'total_acc_x', 'total_acc_y', 'total_acc_z'
]

RANDOM_STATE = 42
print("Configuration loaded.")
print(f"Data root : {DATA_ROOT}")
print(f"Window    : {WINDOW_SIZE} samples ({WINDOW_SIZE/SAMPLE_RATE:.2f}s)")
print(f"Overlap   : {OVERLAP} samples (50%)")

Configuration loaded.
Data root : UCI HAR Dataset
Window    : 128 samples (2.56s)
Overlap   : 64 samples (50%)
